In [ ]:
!pip install -q routellm scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pandas as pd, numpy as np
from tqdm import tqdm

folder = '/content/drive/MyDrive/MLResearch'
df = pd.read_csv(f'{folder}/routerbench_scored_1000_all_routers.csv')
sc = df['bert_score'].values
print(df.shape, [c for c in df.columns if 'score' in c], 'question' in df.columns)

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'placeholder'   # only so the import doesn't fail

from getpass import getpass
from openai import OpenAI
client = OpenAI(api_key=getpass('Paste your NEW key and press Enter: '),
                base_url='https://YOUR-RESOURCE.openai.azure.com/openai/v1/')

import routellm.routers.similarity_weighted.utils as u1
import routellm.routers.routers as u2
import routellm.routers.matrix_factorization.model as u3
for m in (u1, u2, u3):
    m.OPENAI_CLIENT = client      # swap in your real Foundry client

from routellm.controller import Controller
ctrl2 = Controller(routers=['sw_ranking'],
                   strong_model='gpt-4-1106-preview',
                   weak_model='mistralai/Mixtral-8x7B-Instruct-v0.1')
sw = ctrl2.routers['sw_ranking']
print('ready', sw.calculate_strong_win_rate('What is 2+2?'))

In [ ]:
sw_part = f'{folder}/sw_ranking_partial.csv'
done = pd.read_csv(sw_part) if os.path.exists(sw_part) else pd.DataFrame(columns=['row', 'score'])
sw_scores = dict(zip(done.row.astype(int), done.score))
print('already scored:', len(sw_scores))

def save_sw():
    pd.DataFrame({'row': list(sw_scores.keys()), 'score': list(sw_scores.values())}).to_csv(sw_part, index=False)

todo = [i for i in range(len(df)) if i not in sw_scores]
for n, i in enumerate(tqdm(todo, desc='sw_ranking')):
    sw_scores[i] = sw.calculate_strong_win_rate(df.question.iloc[i])
    if (n + 1) % 25 == 0:
        save_sw()
save_sw()
df['sw_ranking_score'] = [sw_scores[i] for i in range(len(df))]
df.to_csv(f'{folder}/routerbench_scored_1000_all_routers.csv', index=False)
print(df[['bert_score', 'mf_score', 'sw_ranking_score']].describe())

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

S, W = df.score_strong.values, df.score_weak.values
CS, CW = df.cost_strong.values, df.cost_weak.values
ds = df.dataset.values
GSM, ARC = 'grade-school-math', 'arc-challenge'
n = len(df); SHARES = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]; B = 2000

tie = np.random.default_rng(0).uniform(0, 1e-6, n)   # random tie-break for dataset-ID scores
R = {'BERT': df.bert_score.values,
     'MF': df.mf_score.values,
     'SW-ranking': df.sw_ranking_score.values,
     'Dataset-ID (GSM8K first)': (ds == GSM) + tie,
     'Dataset-ID (ARC first)': (ds == ARC) + tie}

def evaluate(s, w, cs, cw, b, shares):
    n_ = len(s); order = np.argsort(-b, kind='stable')
    a_s, a_w = s.mean(), w.mean(); out = []
    for sh in shares:
        k = int(round(sh * n_)); ts = np.zeros(n_, bool); ts[order[:k]] = True
        acc = np.where(ts, s, w).mean(); cost = np.where(ts, cs, cw).sum()
        racc = sh * a_s + (1 - sh) * a_w
        rcost = sh * cs.sum() + (1 - sh) * cw.sum()
        q = np.clip((acc - a_w) / (a_s - a_w), 0, 1) if a_s != a_w else 0.0
        crand = q * cs.sum() + (1 - q) * cw.sum()
        out.append((acc, cost, racc, rcost, crand - cost, crand, ts.mean()))
    return np.array(out)

def bootstrap_all(mask, seed=42):
    idx_all = np.where(mask)[0]; rng = np.random.default_rng(seed)
    g = {k: [] for k in R}; sv = {k: [] for k in R}
    for _ in range(B):
        idx = rng.choice(idx_all, len(idx_all))      # one resample shared by ALL routers
        for k, b in R.items():
            r = evaluate(S[idx], W[idx], CS[idx], CW[idx], b[idx], SHARES)
            g[k].append(r[:, 0] - r[:, 2]); sv[k].append(r[:, 4])
    return {k: (np.percentile(g[k], [2.5, 97.5], axis=0),
                np.percentile(sv[k], [2.5, 97.5], axis=0)) for k in R}

rows = []
for name, mask in [('pooled', np.ones(n, bool)), ('GSM8K', ds == GSM), ('ARC', ds == ARC)]:
    idx = np.where(mask)[0]; ci = bootstrap_all(mask)
    for k, b in R.items():
        r = evaluate(S[idx], W[idx], CS[idx], CW[idx], b[idx], SHARES)
        for j, sh in enumerate(SHARES):
            rows.append(dict(subset=name, router=k, share=sh, n=len(idx),
                router_acc=r[j,0], random_acc=r[j,2], gain=r[j,0]-r[j,2],
                gain_lo=ci[k][0][0,j], gain_hi=ci[k][0][1,j],
                router_cost=r[j,1], random_cost=r[j,3],
                saved=r[j,4], saved_lo=ci[k][1][0,j], saved_hi=ci[k][1][1,j],
                pct_saved=100*r[j,4]/r[j,5]))
res = pd.DataFrame(rows)
res.to_csv(f'{folder}/router_comparison_results.csv', index=False)

def mark(lo, hi): return '*' if (lo > 0 or hi < 0) else ''
res['gain_txt'] = [f'{g:+.3f}{mark(l,h)}' for g,l,h in zip(res.gain, res.gain_lo, res.gain_hi)]
res['saved_txt'] = [f'{s:+.2f}{mark(l,h)}' for s,l,h in zip(res.saved, res.saved_lo, res.saved_hi)]
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
print('* = 95% bootstrap interval excludes zero (no multiple-comparison correction)\n')
for name in ['pooled', 'GSM8K', 'ARC']:
    sub = res[res.subset == name]
    print(f'=== {name}: accuracy gain over random, by share sent to GPT-4')
    print(sub.pivot(index='router', columns='share', values='gain_txt').reindex(list(R)), '\n')
sub = res[res.subset == 'pooled']
print('=== pooled: cost saved vs random mix at matching accuracy (USD, negative = router costs more)')
print(sub.pivot(index='router', columns='share', values='saved_txt').reindex(list(R)), '\n')

# AUC: does the score predict "GPT-4 strictly better than Mixtral"?
rng = np.random.default_rng(42); aucs = []
for name, mask in [('pooled', np.ones(n, bool)), ('GSM8K', ds == GSM), ('ARC', ds == ARC)]:
    y = (S[mask] > W[mask]).astype(int)
    for k, b in R.items():
        x = b[mask]; a = roc_auc_score(y, x); bs = []
        for _ in range(1000):
            i = rng.integers(0, len(y), len(y))
            if len(set(y[i])) == 2: bs.append(roc_auc_score(y[i], x[i]))
        lo, hi = np.percentile(bs, [2.5, 97.5])
        aucs.append(dict(subset=name, router=k, auc=round(a,3), ci_low=round(lo,3), ci_high=round(hi,3)))
aucs = pd.DataFrame(aucs); aucs.to_csv(f'{folder}/router_comparison_auc.csv', index=False)
print(aucs.to_string(index=False))

# how much do the routers agree, and how well does each separate GSM8K from ARC?
print('\nSpearman correlation between router scores:')
print(pd.DataFrame({k: R[k] for k in ['BERT','MF','SW-ranking']}).corr(method='spearman').round(3))
for k in ['BERT','MF','SW-ranking']:
    print(k, 'AUC for GSM8K-vs-ARC sorting:', round(roc_auc_score((ds == GSM).astype(int), R[k]), 3))

In [ ]:
import matplotlib.pyplot as plt
p = res[res.subset == 'pooled']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for k, st in [('BERT','o-'), ('MF','s-'), ('SW-ranking','^-'), ('Dataset-ID (GSM8K first)','x--')]:
    q = p[p.router == k]
    axes[0].errorbar(q.share*100, q.gain, yerr=[q.gain-q.gain_lo, q.gain_hi-q.gain],
                     fmt=st, capsize=2, label=k, alpha=.85)
    axes[1].plot(q.share*100, q.saved, st, label=k)
for a in axes: a.axhline(0, color='black', lw=.8); a.set_xlabel('Share sent to GPT-4 (%)'); a.legend()
axes[0].set_ylabel('Accuracy gain over random (95% CI)'); axes[0].set_title('Accuracy gain over random')
axes[1].set_ylabel('Cost saved vs random (USD)\nnegative = router costs more'); axes[1].set_title('Cost saved at matching accuracy')
plt.tight_layout(); plt.savefig(f'{folder}/router_comparison_chart.png', dpi=200); plt.show()

In [ ]:
############################MORE EXPERIMENTS!

In [ ]:
rng = np.random.default_rng(7)
others = ['MF', 'SW-ranking']
out = []
for name, mask in [('pooled', np.ones(n, bool)), ('GSM8K', ds == GSM), ('ARC', ds == ARC)]:
    idx_all = np.where(mask)[0]
    dg = {o: [] for o in others}; ds_ = {o: [] for o in others}
    for _ in range(B):
        idx = rng.choice(idx_all, len(idx_all))       # same resample for all three routers
        rb = evaluate(S[idx], W[idx], CS[idx], CW[idx], R['BERT'][idx], SHARES)
        for o in others:
            ro = evaluate(S[idx], W[idx], CS[idx], CW[idx], R[o][idx], SHARES)
            dg[o].append(rb[:, 0] - ro[:, 0])          # BERT accuracy minus other
            ds_[o].append(rb[:, 4] - ro[:, 4])         # BERT cost saved minus other
    for o in others:
        for metric, arr in [('accuracy', dg[o]), ('cost_saved', ds_[o])]:
            a = np.array(arr); lo, hi = np.percentile(a, [2.5, 97.5], axis=0)
            for j, sh in enumerate(SHARES):
                out.append(dict(subset=name, vs=o, metric=metric, share=sh,
                                bert_minus_other=a[:, j].mean(), lo=lo[j], hi=hi[j],
                                sig=bool(lo[j] > 0 or hi[j] < 0)))
paired = pd.DataFrame(out)
paired.to_csv(f'{folder}/bert_vs_others_paired.csv', index=False)

for metric in ['accuracy', 'cost_saved']:
    for name in ['pooled', 'GSM8K', 'ARC']:
        for o in others:
            s = paired[(paired.metric == metric) & (paired.subset == name) & (paired.vs == o)]
            print(f'{metric:10s} {name:7s} BERT minus {o:10s}: '
                  f'significant at {int(s.sig.sum())} of 9 cut-offs, mean diff {s.bert_minus_other.mean():+.4f}')

In [ ]:
rng = np.random.default_rng(11)
names = ['BERT', 'MF', 'SW-ranking']
gain = {k: [] for k in names}; saved = {k: [] for k in names}
for _ in range(500):
    idx = rng.choice(n, 500, replace=False)        # random 500 of the 1,000 prompts
    for k in names:
        r = evaluate(S[idx], W[idx], CS[idx], CW[idx], R[k][idx], SHARES)
        gain[k].append((r[:, 0] - r[:, 2]).mean())   # mean accuracy gain over the 9 cut-offs
        saved[k].append(r[:, 4].mean())              # mean cost saved over the 9 cut-offs
stab = pd.DataFrame({
    'router': names,
    'mean_gain': [np.mean(gain[k]) for k in names],
    'gain_5th_pct': [np.percentile(gain[k], 5) for k in names],
    'gain_95th_pct': [np.percentile(gain[k], 95) for k in names],
    'pct_samples_gain_positive': [100 * np.mean(np.array(gain[k]) > 0) for k in names],
    'mean_cost_saved': [np.mean(saved[k]) for k in names],
    'pct_samples_cost_saved_positive': [100 * np.mean(np.array(saved[k]) > 0) for k in names]})
stab.to_csv(f'{folder}/stability_500_subsamples.csv', index=False)
print(stab.round(4).to_string(index=False))

In [ ]:
import gc, time
del sw, ctrl2; gc.collect()

from routellm.controller import Controller
ctrl_mf = Controller(routers=['mf'],
                     strong_model='gpt-4-1106-preview',
                     weak_model='mistralai/Mixtral-8x7B-Instruct-v0.1')
mf = ctrl_mf.routers['mf']
mf.calculate_strong_win_rate('warm up')            # exclude one-time setup from the timing
t = time.time()
_ = [mf.calculate_strong_win_rate(q) for q in df.question.iloc[:20]]
print(f'MF: {(time.time()-t)/20:.2f} s per prompt (includes one Foundry embedding call each)')

In [ ]:
from google.colab import files
for f in ['bert_vs_others_paired.csv', 'stability_500_subsamples.csv',
          'router_comparison_results.csv', 'router_comparison_auc.csv',
          'router_comparison_chart.png', 'results_figure_summary.png']:
    p = f'{folder}/{f}'
    if os.path.exists(p): files.download(p)
    else: print('not found yet:', f)

In [ ]:
import matplotlib.pyplot as plt
names = ['BERT', 'MF', 'SW-ranking']
col = {'BERT': '#1f77b4', 'MF': '#ff7f0e', 'SW-ranking': '#2ca02c'}
off = {'BERT': -1.5, 'MF': 0, 'SW-ranking': 1.5}
p = res[res.subset == 'pooled']
fig, ax = plt.subplots(2, 2, figsize=(14, 10))

# (a) pooled accuracy gain over random
for k in names:
    q = p[p.router == k]
    ax[0,0].errorbar(q.share*100 + off[k], q.gain, yerr=[q.gain-q.gain_lo, q.gain_hi-q.gain],
                     fmt='o-', capsize=2, color=col[k], label=k, alpha=.9)
ax[0,0].axhline(0, color='black', lw=.8)
ax[0,0].set(title='(a) Accuracy gain over random (pooled, 95% CI)',
            xlabel='Share sent to GPT-4 (%)', ylabel='Mean score gain')
ax[0,0].legend()

# (b) pooled cost saved at matching accuracy
for k in names:
    q = p[p.router == k]
    ax[0,1].errorbar(q.share*100 + off[k], q.saved, yerr=[q.saved-q.saved_lo, q.saved_hi-q.saved],
                     fmt='o-', capsize=2, color=col[k], label=k, alpha=.9)
ax[0,1].axhline(0, color='black', lw=.8)
ax[0,1].set(title='(b) Cost saved vs random at matching accuracy (pooled)',
            xlabel='Share sent to GPT-4 (%)', ylabel='USD per 1,000 prompts\nnegative = router costs more')
ax[0,1].legend()

# (c) AUC by subset
subs = ['pooled', 'GSM8K', 'ARC']; w = 0.26
for j, k in enumerate(names):
    a = aucs[aucs.router == k].set_index('subset').loc[subs]
    ax[1,0].bar(np.arange(3) + (j-1)*w, a.auc, w, yerr=[a.auc-a.ci_low, a.ci_high-a.auc],
                capsize=3, color=col[k], label=k)
ax[1,0].axhline(0.5, color='black', lw=.8, ls='--')
ax[1,0].set_xticks(range(3)); ax[1,0].set_xticklabels(['Pooled', 'GSM8K only', 'ARC only'])
ax[1,0].set(ylim=(0.35, 0.85), ylabel='AUC (0.5 = no signal)',
            title='(c) Does the score predict when GPT-4 beats Mixtral?')
ax[1,0].legend()

# (d) stability across 500 random subsamples
s = stab.set_index('router').loc[names]
bars = ax[1,1].bar(names, s.mean_cost_saved, color=[col[k] for k in names])
ax[1,1].axhline(0, color='black', lw=.8)
for b, pct in zip(bars, s.pct_samples_cost_saved_positive):
    ax[1,1].annotate(f'{pct:.1f}% of samples\nsaved money', (b.get_x()+b.get_width()/2, b.get_height()),
                     ha='center', va='top', xytext=(0, -4), textcoords='offset points', fontsize=9)
ax[1,1].set(ylabel='Mean cost saved (USD)', title='(d) Cost saved at matching accuracy,\nacross 500 random 500-prompt subsamples')

plt.suptitle('RouteLLM routers on 1,000 RouterBench prompts (600 GSM8K + 400 ARC-Challenge)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{folder}/results_figure_summary.png', dpi=200)
plt.show()

In [ ]:
import time, gc, os, torch
os.environ.setdefault('OPENAI_API_KEY', 'placeholder')
from routellm.controller import Controller
print('GPU available:', torch.cuda.is_available())

timing = {}
def time_router(name, n):
    c = Controller(routers=[name], strong_model='gpt-4-1106-preview',
                   weak_model='mistralai/Mixtral-8x7B-Instruct-v0.1')
    r = c.routers[name]; r.calculate_strong_win_rate('warm up')   # exclude one-time setup
    t = time.time()
    for q in df.question.iloc[:n]: r.calculate_strong_win_rate(q)
    timing[name] = (time.time() - t) / n
    del c, r; gc.collect()

time_router('bert', 50)
time_router('mf', 30)
time_router('sw_ranking', 10)
for k, v in timing.items():
    print(f'{k:12s} {v:.3f} s per prompt -> {v*1000/60:.1f} min per 1,000 prompts')

In [ ]:
# B: external dependence. Embedding routers make one API call per prompt; BERT makes none.
chars = df.question.str.len().sum()
tokens_est = chars / 4                       # rough rule of thumb, not exact tokenization
print(f'B) {n} prompts, about {tokens_est:,.0f} tokens (rough estimate)')
print('   API calls needed: BERT 0 | MF', n, '| SW-ranking', n)
print('   Check Azure/OpenAI pricing for text-embedding-3-small; at this size the cost is tiny,')
print('   so the real advantage of BERT is no key/quota/network dependence, not money saved.')

# C: do the routers send the same prompts to GPT-4? (top 50% by score)
from itertools import combinations
k = n // 2
top = {name: set(np.argsort(-R[name], kind='stable')[:k]) for name in ['BERT', 'MF', 'SW-ranking']}
print('\nC) overlap of the top-50% sets (random expectation is 0.50)')
for a, b in combinations(top, 2):
    inter = len(top[a] & top[b])
    print(f'   {a} vs {b}: {inter / k:.3f} of prompts shared, Jaccard {inter / len(top[a] | top[b]):.3f}')

# same check inside each dataset, so GSM8K-vs-ARC sorting cannot inflate agreement
from scipy.stats import spearmanr
print('\n   Within-dataset Spearman correlation between routers')
for d, lab in [(GSM, 'GSM8K'), (ARC, 'ARC')]:
    m = ds == d
    for a, b in combinations(['BERT', 'MF', 'SW-ranking'], 2):
        print(f'   {lab:6s} {a} vs {b}: {spearmanr(R[a][m], R[b][m])[0]:.3f}')

In [ ]:
for name in ['df', 'R', 'res', 'aucs', 'stab', 'folder']:
    print(name, 'OK' if name in globals() else 'MISSING')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pandas as pd, numpy as np
folder = '/content/drive/MyDrive/MLResearch'
df = pd.read_csv(f'{folder}/routerbench_scored_1000_all_routers.csv')
n, ds = len(df), df.dataset.values
GSM, ARC = 'grade-school-math', 'arc-challenge'
R = {'BERT': df.bert_score.values, 'MF': df.mf_score.values,
     'SW-ranking': df.sw_ranking_score.values}
print(n, list(R))

In [ ]:
# B: external dependence
chars = df.question.str.len().sum()
tokens_est = chars / 4                       # rough rule of thumb, not exact tokenization
print(f'B) {n} prompts, about {tokens_est:,.0f} tokens (rough estimate)')
print('   API calls needed: BERT 0 | MF', n, '| SW-ranking', n)
print('   At this size the embedding cost is tiny, so the real advantage of BERT is')
print('   no key/quota/network dependence, not money saved.')

# C: do the routers send the same prompts to GPT-4? (top 50% by score)
from itertools import combinations
from scipy.stats import spearmanr
k = n // 2
top = {name: set(np.argsort(-R[name], kind='stable')[:k]) for name in R}
print('\nC) overlap of the top-50% sets (random expectation is 0.50)')
for a, b in combinations(top, 2):
    inter = len(top[a] & top[b])
    print(f'   {a} vs {b}: {inter / k:.3f} of prompts shared, Jaccard {inter / len(top[a] | top[b]):.3f}')

print('\n   Within-dataset Spearman correlation between routers')
for d, lab in [(GSM, 'GSM8K'), (ARC, 'ARC')]:
    m = ds == d
    for a, b in combinations(R, 2):
        print(f'   {lab:6s} {a} vs {b}: {spearmanr(R[a][m], R[b][m])[0]:.3f}')

In [ ]:
res  = pd.read_csv(f'{folder}/router_comparison_results.csv')
aucs = pd.read_csv(f'{folder}/router_comparison_auc.csv')
stab = pd.read_csv(f'{folder}/stability_500_subsamples.csv')
print(res.shape, aucs.shape, stab.shape)
print(os.listdir(folder))

In [ ]:
import os, time, gc
import pandas as pd
os.environ.setdefault('OPENAI_API_KEY', 'placeholder')

folder = '/content/drive/MyDrive/MLResearch'
df = pd.read_csv(f'{folder}/routerbench_scored_1000_all_routers.csv')

from routellm.controller import Controller
c = Controller(routers=['bert'], strong_model='gpt-4-1106-preview',
               weak_model='mistralai/Mixtral-8x7B-Instruct-v0.1')
r = c.routers['bert']; r.calculate_strong_win_rate('warm up')
t = time.time()
for q in df.question.iloc[:50]: r.calculate_strong_win_rate(q)
print(f'BERT: {(time.time()-t)/50:.3f} s per prompt')

In [ ]:
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt

folder = '/content/drive/MyDrive/MLResearch'
stab = pd.read_csv(f'{folder}/stability_500_subsamples.csv').set_index('router')
aucs = pd.read_csv(f'{folder}/router_comparison_auc.csv')
df   = pd.read_csv(f'{folder}/routerbench_scored_1000_all_routers.csv')
names = ['BERT', 'MF', 'SW-ranking']
col = {'BERT': '#1f77b4', 'MF': '#ff7f0e', 'SW-ranking': '#2ca02c'}
n = len(df)

# Hard-coded inputs (say where they came from in the paper):
api_calls = {'BERT': 0, 'MF': n, 'SW-ranking': n}                  # from the code path, B experiment
sec = {'BERT': 55*60/8920, 'MF': np.nan, 'SW-ranking': 3.69}       # BERT: earlier full run (8,920 prompts, ~55 min, CPU)
                                                                   # SW-ranking: timed in this study; MF: not measured
arc = aucs[aucs.subset == 'ARC'].set_index('router').loc[names]

fig, ax = plt.subplots(2, 3, figsize=(16, 9))

# (a) API calls
ax[0,0].bar(names, [api_calls[k] for k in names], color=[col[k] for k in names])
for i, k in enumerate(names): ax[0,0].text(i, api_calls[k], str(api_calls[k]), ha='center', va='bottom')
ax[0,0].set(title='(a) External API calls needed\nfor 1,000 prompts (lower = better)', ylabel='Calls')

# (b) speed
vals = [0 if np.isnan(sec[k]) else sec[k] for k in names]
ax[0,1].bar(names, vals, color=[col[k] for k in names])
for i, k in enumerate(names):
    ax[0,1].text(i, vals[i], 'not measured' if np.isnan(sec[k]) else f'{sec[k]:.2f} s',
                 ha='center', va='bottom')
ax[0,1].set(title='(b) Seconds per prompt on CPU (lower = better)\nBERT: earlier full run; SW-ranking: this study', ylabel='Seconds')

# (c) accuracy gain over random
m = stab.loc[names, 'mean_gain']
err = [m - stab.loc[names, 'gain_5th_pct'], stab.loc[names, 'gain_95th_pct'] - m]
ax[0,2].bar(names, m, yerr=err, capsize=5, color=[col[k] for k in names])
ax[0,2].axhline(0, color='black', lw=.8)
ax[0,2].set(title='(c) Mean accuracy gain over random\n(5th-95th percentile over 500 subsamples)', ylabel='Score gain')

# (d) cost saved at matching accuracy
c = stab.loc[names, 'mean_cost_saved']
ax[1,0].bar(names, c, color=[col[k] for k in names])
ax[1,0].axhline(0, color='black', lw=.8)
for i, k in enumerate(names):
    ax[1,0].text(i, c[k], f'{c[k]:+.2f}', ha='center', va='top' if c[k] < 0 else 'bottom')
ax[1,0].set(title='(d) Cost saved vs random at matching accuracy\n(negative = costs more; BERT is worst)', ylabel='USD')

# (e) ARC AUC
ax[1,1].bar(names, arc.auc, yerr=[arc.auc - arc.ci_low, arc.ci_high - arc.auc],
            capsize=5, color=[col[k] for k in names])
ax[1,1].axhline(0.5, color='black', ls='--', lw=.8)
ax[1,1].set(ylim=(0.4, 0.85), title='(e) ARC: does the score predict when GPT-4 is better?\n(0.5 = no signal)', ylabel='AUC')

# (f) scorecard
G, R_, N = '#c8e6c9', '#ffcdd2', '#eeeeee'
cost_best, cost_worst = c.idxmax(), c.idxmin()
rows = [
  ('No API key / service',      [str(api_calls[k]) + ' calls' for k in names],
                                [G if k == 'BERT' else R_ for k in names]),
  ('CPU time per prompt',       ['n/a' if np.isnan(sec[k]) else f'{sec[k]:.2f} s' for k in names],
                                [G if k == 'BERT' else (N if np.isnan(sec[k]) else R_) for k in names]),
  ('Accuracy gain vs random',   [f'{m[k]:+.4f}' for k in names], [N]*3),
  ('Cost saved at same acc.',   [f'{c[k]:+.2f}' for k in names],
                                [G if k == cost_best else (R_ if k == cost_worst else N) for k in names]),
  ('ARC AUC',                   [f'{arc.auc[k]:.2f}' for k in names], [N]*3)]
ax[1,2].axis('off')
tbl = ax[1,2].table(cellText=[r[1] for r in rows], rowLabels=[r[0] for r in rows],
                    colLabels=names, cellColours=[r[2] for r in rows], loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1, 2.2)
ax[1,2].set_title('(f) Scorecard: green = best, red = worst, grey = tie / not measured')

plt.suptitle('Why BERT: a fit-for-constraints comparison of the three RouteLLM routers we ran\n'
             '(1,000 RouterBench prompts; accuracy is a tie, cost is BERT\'s weakness)', fontsize=13)
plt.tight_layout()
out = f'{folder}/why_bert_criteria_figure.png'
plt.savefig(out, dpi=200); plt.show()

# the same table as a CSV: this is your auditable record
pd.DataFrame({r[0]: r[1] for r in rows}, index=names).T.to_csv(f'{folder}/why_bert_scorecard.csv')
print('saved:', out)

In [ ]:
import numpy as np, pandas as pd
folder = '/content/drive/MyDrive/MLResearch'
df = pd.read_csv(f'{folder}/routerbench_scored_1000_all_routers.csv')
S, W, CS, CW, ds = df.score_strong.values, df.score_weak.values, df.cost_strong.values, df.cost_weak.values, df.dataset.values
R = {'BERT': df.bert_score.values, 'MF': df.mf_score.values, 'SW-ranking': df.sw_ranking_score.values}

# Held-out cut-off: threshold from half A (top 50% -> GPT-4), applied to half B
rng = np.random.default_rng(3); rows = []
for rep in range(300):
    A, Bi = [], []
    for d in np.unique(ds):                       # stratified split so both halves keep the dataset mix
        ii = rng.permutation(np.where(ds == d)[0]); h = len(ii) // 2
        A += list(ii[:h]); Bi += list(ii[h:])
    A, Bi = np.array(A), np.array(Bi)
    for k, b in R.items():
        ts = b[Bi] > np.quantile(b[A], 0.5); share = ts.mean()
        acc = np.where(ts, S[Bi], W[Bi]).mean()
        cost = np.where(ts, CS[Bi], CW[Bi]).sum()
        racc = share * S[Bi].mean() + (1 - share) * W[Bi].mean()
        rcost = share * CS[Bi].sum() + (1 - share) * CW[Bi].sum()
        rows.append(dict(router=k, share_error=abs(share - 0.5), gain=acc - racc, cost_vs_random=rcost - cost))
ho = pd.DataFrame(rows).groupby('router').agg(
    mean_share_error=('share_error', 'mean'),
    mean_gain=('gain', 'mean'),
    pct_splits_gain_positive=('gain', lambda x: 100 * (x > 0).mean()),
    mean_cost_vs_random_same_share=('cost_vs_random', 'mean')).round(4)
print('Held-out cut-off (threshold set on one half, applied to the other; 300 splits)')
print(ho)
ho.to_csv(f'{folder}/heldout_cutoff_results.csv')

# Share of the possible gain captured at a 50% cut-off (pooled)
oracle_acc = np.maximum(S, W).mean()
rand_acc = 0.5 * S.mean() + 0.5 * W.mean()
print(f'\nRandom {rand_acc:.4f} -> oracle {oracle_acc:.4f}: room for gain = {oracle_acc - rand_acc:.4f}')
order_n = len(S)
for k, b in R.items():
    ts = np.zeros(order_n, bool); ts[np.argsort(-b, kind="stable")[:order_n // 2]] = True
    g = np.where(ts, S, W).mean() - rand_acc
    print(f'{k:10s} gain {g:+.4f} = {100 * g / (oracle_acc - rand_acc):.1f}% of the possible gain')

In [ ]:
import os, time, psutil
import numpy as np, pandas as pd
os.environ.setdefault('OPENAI_API_KEY', 'placeholder')   # no real key needed for BERT

from google.colab import drive
drive.mount('/content/drive')
folder = '/content/drive/MyDrive/MLResearch'
df = pd.read_csv(f'{folder}/routerbench_scored_1000_all_routers.csv')
qs = list(df.question.iloc[:50])

import torch
proc = psutil.Process()
ram_before = proc.memory_info().rss / 1e9

t = time.time()
from routellm.controller import Controller
c = Controller(routers=['bert'], strong_model='gpt-4-1106-preview',
               weak_model='mistralai/Mixtral-8x7B-Instruct-v0.1')
bert = c.routers['bert']; bert.calculate_strong_win_rate('warm up')
load_s = time.time() - t
ram_after = proc.memory_info().rss / 1e9

t = time.time(); run1 = np.array([bert.calculate_strong_win_rate(q) for q in qs]); per = (time.time() - t) / len(qs)
run2 = np.array([bert.calculate_strong_win_rate(q) for q in qs])
saved = df.bert_score.values[:50]

print('GPU used:', torch.cuda.is_available())
print(f'Load time (incl. import): {load_s:.1f} s')
print(f'Process memory: {ram_before:.2f} GB before -> {ram_after:.2f} GB after loading BERT')
print(f'Time per prompt: {per:.3f} s -> {per*1000/60:.1f} min per 1,000 prompts')
print('Repeat run, max difference:', np.abs(run1 - run2).max())
print('Versus scores saved earlier (different session), max difference:', np.abs(run1 - saved).max())

pd.DataFrame([dict(router='BERT', gpu=torch.cuda.is_available(), load_s=load_s, ram_gb=ram_after,
                   s_per_prompt=per, repeat_max_diff=np.abs(run1 - run2).max(),
                   vs_saved_max_diff=np.abs(run1 - saved).max())]
            ).to_csv(f'{folder}/bert_footprint_measured.csv', index=False)

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

folder = '/content/drive/MyDrive/MLResearch'
stab = pd.read_csv(f'{folder}/stability_500_subsamples.csv').set_index('router')
names = ['BERT', 'MF', 'SW-ranking']
col = {'BERT': '#1f77b4', 'MF': '#ff7f0e', 'SW-ranking': '#2ca02c'}

# ---- measured values (source in comments); np.nan = not measured ----
api_calls = {'BERT': 0, 'MF': 1000, 'SW-ranking': 1000}          # per 1,000 prompts, from how each router works
sec       = {'BERT': 0.305, 'MF': np.nan, 'SW-ranking': 3.69}    # s/prompt, Colab CPU; SW-ranking includes an API call
repeat    = {'BERT': 1.5e-8}                                     # max score difference vs earlier session

def bar(ax, d, title, ylabel, fmt):
    vals = [0 if np.isnan(d[k]) else d[k] for k in names]
    ax.bar(names, vals, color=[col[k] for k in names])
    top = max(vals) if max(vals) > 0 else 1
    for i, k in enumerate(names):
        txt = 'not\nmeasured' if np.isnan(d[k]) else fmt.format(d[k])
        ax.text(i, vals[i] + top * 0.02, txt, ha='center', va='bottom', fontsize=9)
    ax.set_ylim(0, top * 1.25); ax.set_title(title, fontsize=10); ax.set_ylabel(ylabel)

fig, ax = plt.subplots(2, 3, figsize=(16, 9))

# (a) API calls and (b) speed
bar(ax[0,0], api_calls, '(a) External API calls per 1,000 prompts\n(lower = better)', 'Calls', '{:.0f}')
bar(ax[0,1], sec, '(b) Seconds per prompt on CPU (lower = better)\nBERT and SW-ranking measured; MF not measured',
    'Seconds', '{:.3f} s')

# (c) memory: only BERT was measured, so show a text note instead of a one-bar chart
ax[0,2].axis('off')
ax[0,2].set_title('(c) Memory added by loading the router', fontsize=10)
ax[0,2].text(0.02, 0.55,
    'BERT: about 0.69 GB added\n(1.54 -> 2.23 GB process memory),\nloaded in about 8 s.\n\n'
    'MF and SW-ranking: not measured,\nso no comparison is made.', fontsize=10, va='center')

# (d) repeatability, as text because 1.5e-8 is invisible on a bar axis
ax[1,0].axis('off')
ax[1,0].set_title('(d) Repeatability', fontsize=10)
ax[1,0].text(0.02, 0.55,
    'BERT: identical scores on a repeat run in the\nsame session (difference 0.0).\n\n'
    f'Versus scores from an earlier session:\nmax difference {repeat["BERT"]:.1e} (rounding noise).\n\n'
    'MF and SW-ranking: not tested.', fontsize=10, va='center')

# (e) accuracy gain: a tie
m = stab.loc[names, 'mean_gain']
err = [m - stab.loc[names, 'gain_5th_pct'], stab.loc[names, 'gain_95th_pct'] - m]
ax[1,1].bar(names, m, yerr=err, capsize=5, color=[col[k] for k in names])
ax[1,1].axhline(0, color='black', lw=.8)
ax[1,1].set_title('(e) Accuracy gain over random: a tie\n(paired tests found no detectable difference)', fontsize=10)
ax[1,1].set_ylabel('Mean score gain (5th-95th pct)')

# (f) cost trade-off, BERT's weakness (axis extended so labels do not hit the border)
c = stab.loc[names, 'mean_cost_saved']
ax[1,2].bar(names, c, color=[col[k] for k in names]); ax[1,2].axhline(0, color='black', lw=.8)
for i, k in enumerate(names): ax[1,2].text(i, c[k], f'{c[k]:+.2f}', ha='center', va='top')
ax[1,2].set_ylim(c.min() * 1.25, 0.02)
ax[1,2].set_title('(f) Cost saved vs random at matching accuracy\n(negative = costs more; BERT is worst)', fontsize=10)
ax[1,2].set_ylabel('USD')

plt.suptitle('BERT: practical advantages and the cost trade-off (RouteLLM routers run on 1,000 RouterBench prompts)',
             fontsize=13)
plt.tight_layout()
plt.savefig(f'{folder}/why_bert_practical_figure.png', dpi=200); plt.show()
print('saved', f'{folder}/why_bert_practical_figure.png')

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
folder = '/content/drive/MyDrive/MLResearch'
res = pd.read_csv(f'{folder}/router_comparison_results.csv')
names = ['BERT', 'MF', 'SW-ranking']
col = {'BERT': '#1f77b4', 'MF': '#ff7f0e', 'SW-ranking': '#2ca02c'}

fig, ax = plt.subplots(1, 3, figsize=(16, 4.8), sharey=True)
for a, sub in zip(ax, ['pooled', 'GSM8K', 'ARC']):
    for k in names:
        q = res[(res.subset == sub) & (res.router == k)].sort_values('share')
        a.errorbar(q.share * 100, q.saved, yerr=[q.saved - q.saved_lo, q.saved_hi - q.saved],
                   fmt='o-', capsize=2, color=col[k], label=k, alpha=.85)
    a.axhline(0, color='black', lw=.8)
    a.set_title(f'{sub}: cost saved vs random at matching accuracy'); a.set_xlabel('Share sent to GPT-4 (%)')
ax[0].set_ylabel('USD (negative = costs more)'); ax[0].legend()
plt.suptitle('Cost at matching accuracy, pooled and within each dataset (95% bootstrap intervals)')
plt.tight_layout(); plt.savefig(f'{folder}/cost_by_dataset_figure.png', dpi=200); plt.show()

# mean over cut-offs, for a one-line summary per router and subset
print(res[res.router.isin(names)].groupby(['subset', 'router']).saved.mean().round(3).unstack())

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
folder = '/content/drive/MyDrive/MLResearch'
d = pd.read_csv(f'{folder}/arc_0shot_vs_5shot_robustness.csv')
d = d[d.subset == 'pooled']
routers = ['BERT', 'MF', 'SW-ranking', 'Dataset-ID (GSM8K first)']
style = {'ARC 0-shot': ('o', '#eb6834', -0.12), 'ARC 5-shot': ('s', '#2a78d6', 0.12)}

fig, ax = plt.subplots(figsize=(8, 4.5))
for cond, (mk, col, off) in style.items():
    for i, r in enumerate(routers):
        q = d[(d.condition == cond) & (d.router == r)]
        if q.empty: continue
        q = q.iloc[0]
        ax.errorbar(q.saved, i + off, xerr=[[q.saved - q.saved_lo], [q.saved_hi - q.saved]],
                    fmt=mk, color=col, capsize=3, label=cond if i == 0 else None)
ax.axvline(0, color='black', ls='--', lw=.8)
ax.set_yticks(range(len(routers))); ax.set_yticklabels(routers); ax.invert_yaxis()
ax.set_xlabel('USD saved vs random at matching accuracy\n(negative = router costs more; mean over 9 cut-offs, 95% bootstrap CI)')
ax.set_title('Pooled cost result: 0-shot vs 5-shot ARC')
ax.legend(); plt.tight_layout()
plt.savefig(f'{folder}/cost_0shot_vs_5shot_figure.png', dpi=200); plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
print([f for f in os.listdir('/content/drive/MyDrive/MLResearch') if 'robust' in f or '5shot' in f])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, matplotlib.pyplot as plt

folder = '/content/drive/MyDrive/MLResearch'
d = pd.read_csv(f'{folder}/arc_0shot_vs_5shot_robustness.csv')
d = d[d.subset == 'pooled']
print(d[['condition', 'router', 'saved', 'saved_lo', 'saved_hi']].round(2).to_string(index=False))

routers = ['BERT', 'MF', 'SW-ranking', 'Dataset-ID (GSM8K first)']
style = {'ARC 0-shot': ('o', '#eb6834', -0.12), 'ARC 5-shot': ('s', '#2a78d6', 0.12)}

fig, ax = plt.subplots(figsize=(8, 4.5))
for cond, (mk, col, off) in style.items():
    for i, r in enumerate(routers):
        q = d[(d.condition == cond) & (d.router == r)].iloc[0]
        ax.errorbar(q.saved, i + off, xerr=[[q.saved - q.saved_lo], [q.saved_hi - q.saved]],
                    fmt=mk, color=col, capsize=3, label=cond if i == 0 else None)
ax.axvline(0, color='black', ls='--', lw=.8)
ax.set_yticks(range(len(routers)))
ax.set_yticklabels(['BERT', 'MF', 'SW-ranking', 'Dataset label only'])
ax.invert_yaxis()
ax.set_xlabel('USD saved vs random at matching accuracy\n(negative = router costs more; mean over 9 cut-offs, 95% bootstrap CI)')
ax.set_title('Pooled cost result: 0-shot vs 5-shot ARC (997 prompts)')
ax.legend(); plt.tight_layout()
plt.savefig(f'{folder}/cost_0shot_vs_5shot_figure.png', dpi=200); plt.show()

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
folder = '/content/drive/MyDrive/MLResearch'
d = pd.read_csv(f'{folder}/arc_0shot_vs_5shot_robustness.csv')
d = d[d.subset == 'pooled']
routers = ['BERT', 'MF', 'SW-ranking', 'Dataset-ID (GSM8K first)']
style = {'ARC 0-shot': ('o', '#eb6834', -0.12), 'ARC 5-shot': ('s', '#2a78d6', 0.12)}

fig, ax = plt.subplots(figsize=(8, 4.5))
for cond, (mk, col, off) in style.items():
    for i, r in enumerate(routers):
        q = d[(d.condition == cond) & (d.router == r)]
        if q.empty: continue
        q = q.iloc[0]
        ax.errorbar(q.saved, i + off, xerr=[[q.saved - q.saved_lo], [q.saved_hi - q.saved]],
                    fmt=mk, color=col, capsize=3, label=cond if i == 0 else None)
ax.axvline(0, color='black', ls='--', lw=.8)
ax.set_yticks(range(len(routers))); ax.set_yticklabels(routers); ax.invert_yaxis()
ax.set_xlabel('USD saved vs random at matching accuracy\n(negative = router costs more; mean over 9 cut-offs, 95% bootstrap CI)')
ax.set_title('Pooled cost result: 0-shot vs 5-shot ARC')
ax.legend(); plt.tight_layout()
plt.savefig(f'{folder}/cost_0shot_vs_5shot_figure.png', dpi=200); plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download

folder = '/content/drive/MyDrive/MLResearch'
M, G = 'mistralai/mixtral-8x7b-chat', 'gpt-4-1106-preview'
GSM, ARC = 'grade-school-math', 'arc-challenge'
SHARES = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

def evaluate(s, w, cs, cw, b, shares):
    n_ = len(s); order = np.argsort(-b, kind='stable')
    a_s, a_w = s.mean(), w.mean(); out = []
    for sh in shares:
        k = int(round(sh * n_)); ts = np.zeros(n_, bool); ts[order[:k]] = True
        acc = np.where(ts, s, w).mean(); cost = np.where(ts, cs, cw).sum()
        racc = sh * a_s + (1 - sh) * a_w
        rcost = sh * cs.sum() + (1 - sh) * cw.sum()
        q = np.clip((acc - a_w) / (a_s - a_w), 0, 1) if a_s != a_w else 0.0
        crand = q * cs.sum() + (1 - q) * cw.sum()
        out.append((acc, cost, racc, rcost, crand - cost, crand, ts.mean()))
    return np.array(out)

# build 0-shot and 5-shot tables (same logic as before)
base = pd.read_csv(f'{folder}/routerbench_scored_1000_all_routers.csv').reset_index(drop=True)
f = pd.read_pickle(hf_hub_download('withmartian/routerbench', 'routerbench_5shot.pkl', repo_type='dataset'))
f5 = f.drop_duplicates('sample_id').set_index('sample_id')
d5 = base.copy()
arc = (d5.dataset == ARC).values
ids = d5.sample_id[arc]
d5.loc[arc, 'score_strong'] = pd.to_numeric(f5.loc[ids, G], errors='coerce').values
d5.loc[arc, 'score_weak']   = pd.to_numeric(f5.loc[ids, M], errors='coerce').values
d5.loc[arc, 'cost_strong']  = f5.loc[ids, G + '|total_cost'].astype(float).values
d5.loc[arc, 'cost_weak']    = f5.loc[ids, M + '|total_cost'].astype(float).values

cols = ['score_strong', 'score_weak', 'cost_strong', 'cost_weak']
bad_ids = d5.loc[d5[cols].isna().any(axis=1), 'sample_id']
keep = ~d5.sample_id.isin(bad_ids).values
b0, b5 = base[keep].reset_index(drop=True), d5[keep].reset_index(drop=True)
print(f'dropped {(~keep).sum()} prompts | analysing {len(b0)}')

tie = np.random.default_rng(0).uniform(0, 1e-6, len(b0))
R = {'BERT': b0.bert_score.values, 'MF': b0.mf_score.values,
     'SW-ranking': b0.sw_ranking_score.values,
     'Dataset label only': (b0.dataset.values == GSM) + tie}
col = {'BERT': '#1f77b4', 'MF': '#ff7f0e', 'SW-ranking': '#2ca02c', 'Dataset label only': '#7f7f7f'}
fmt = {'BERT': 'o-', 'MF': 's-', 'SW-ranking': '^-', 'Dataset label only': 'x--'}

def curves(d, mask, B=500, seed=42):
    S, W, CS, CW = [d[c].values.astype(float) for c in cols]
    ia = np.where(mask)[0]; rng = np.random.default_rng(seed)
    boots = [rng.choice(ia, len(ia)) for _ in range(B)]
    out = {}
    for k, b in R.items():
        pt = evaluate(S[ia], W[ia], CS[ia], CW[ia], b[ia], SHARES)[:, 4]
        bs = np.array([evaluate(S[i], W[i], CS[i], CW[i], b[i], SHARES)[:, 4] for i in boots])
        out[k] = (pt, np.percentile(bs, 2.5, axis=0), np.percentile(bs, 97.5, axis=0))
    return out

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharey=True, sharex=True)
for r_i, (cond, d) in enumerate([('ARC 0-shot', b0), ('ARC 5-shot', b5)]):
    ds = d.dataset.values
    for c_i, (sub, mask) in enumerate([('Pooled', np.ones(len(d), bool)), ('GSM8K', ds == GSM), ('ARC', ds == ARC)]):
        a = axes[r_i, c_i]
        for k, (pt, lo, hi) in curves(d, mask).items():
            a.errorbar(np.array(SHARES) * 100, pt, yerr=[pt - lo, hi - pt], fmt=fmt[k],
                       capsize=2, color=col[k], label=k, alpha=.85)
        a.axhline(0, color='black', lw=.8)
        a.set_title(f'{cond}: {sub}')
        if r_i == 1: a.set_xlabel('Share sent to GPT-4 (%)')
    axes[r_i, 0].set_ylabel('USD saved vs random\n(negative = router costs more)')
axes[0, 0].legend()
plt.suptitle('Cost at matching accuracy: 0-shot vs 5-shot ARC (997 prompts, 95% bootstrap intervals)')
plt.tight_layout(); plt.savefig(f'{folder}/cost_curves_0shot_vs_5shot.png', dpi=200); plt.show()

In [ ]:
import numpy as np, matplotlib.pyplot as plt
x = np.array(SHARES) * 100
col = {'BERT': '#1f77b4', 'MF': '#ff7f0e', 'SW-ranking': '#2ca02c', 'Dataset label only': '#7f7f7f'}

def draw(ax, cr, band, title):
    for k, (pt, lo, hi) in cr.items():
        if k not in band and k == 'Dataset label only' and 'Dataset label only' not in cr: continue
        thick = k in band
        ax.plot(x, pt, '--' if k == 'Dataset label only' else '-', color=col[k],
                lw=2.4 if thick else 1.2, alpha=1 if thick else .7, label=k)
        if thick: ax.fill_between(x, lo, hi, color=col[k], alpha=.15)
    ax.axhline(0, color='black', lw=.8); ax.set_title(title)
    ax.set_xlabel('Share sent to GPT-4 (%)'); ax.grid(alpha=.2)

# Figure A: pooled
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for a, (cond, d) in zip(axes, [('ARC 0-shot', b0), ('ARC 5-shot', b5)]):
    draw(a, curves(d, np.ones(len(d), bool)), band=['BERT', 'Dataset label only'], title=cond)
axes[0].set_ylabel('USD saved vs random at matching accuracy\n(negative = router costs more)')
axes[0].legend(loc='lower left')
plt.suptitle('Pooled cost at matching accuracy (997 prompts; shaded = 95% bootstrap interval)')
plt.tight_layout(); plt.savefig(f'{folder}/figA_pooled_cost.png', dpi=200); plt.show()

# Figure B: ARC only, own y-scale, no dataset-label line
R_backup = R; R = {k: v for k, v in R_backup.items() if k != 'Dataset label only'}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for a, (cond, d) in zip(axes, [('ARC 0-shot', b0), ('ARC 5-shot', b5)]):
    draw(a, curves(d, d.dataset.values == ARC), band=['BERT', 'SW-ranking'], title=f'{cond}: ARC prompts only')
axes[0].set_ylabel('USD saved vs random at matching accuracy')
axes[0].legend(loc='upper left')
plt.suptitle('Within-ARC cost at matching accuracy (shaded = 95% bootstrap interval)')
plt.tight_layout(); plt.savefig(f'{folder}/figB_arc_cost.png', dpi=200); plt.show()
R = R_backup